# The first model: agent picks

Fit a model to the 39 numbers from notebook 03 and see whether it beats guessing. Notebook 03
found no single agent predicts winning, so **expect close to a coin flip**. A high score would
mean a bug.

**The baselines to beat:**

| Guess | Right, over the whole season |
|---|---|
| Always team A | 51.1% |
| Whoever picked the map; team A when nobody did | 54.6% |

The rule needs its fallback because 225 of the 1,272 maps (18%) are deciders that nobody
picks. A model answers on every map, so the rule it's measured against must too. The often
quoted 53.8% covers only the maps someone picked.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

from dataset import build_map_table
from features import build_features

maps = build_map_table()
X, y = build_features(maps)

print(f"{len(maps):,} maps, each described by {X.shape[1]} numbers")

1,272 maps, each described by 39 numbers


## 1. Train, validation and test

Comparing many attempts on the test set and keeping the best would tune the model to those
matches. So the season is split three ways:

| Part | Used for |
|---|---|
| **Train** | learning |
| **Validation** | comparing attempts |
| **Test** | untouched until a final model is chosen |

In [2]:
from dataset import SPLIT_DATE, split_season

learn, test = split_season(maps)

timeline = (
    maps.groupby("tournament")
    .agg(first_map=("played_at", "min"), last_map=("played_at", "max"), maps=("map", "size"))
    .sort_values("first_map")
)
timeline["half"] = np.where(timeline["first_map"] < SPLIT_DATE, "learn from", "test on")

print(timeline.to_string(formatters={
    "first_map": lambda d: d.strftime("%d %b"),
    "last_map": lambda d: d.strftime("%d %b"),
}))

print(f"\nsplit at {SPLIT_DATE:%d %b %Y}")
print(f"learn from: {len(learn):,} maps      test on: {len(test):,} maps")
print(f"tournaments split across both halves: {len(set(learn['tournament']) & set(test['tournament']))}")

                              first_map last_map  maps        half
tournament                                                        
VCT 2025: China Kickoff          11 Jan   25 Jan    56  learn from
VCT 2025: EMEA Kickoff           15 Jan   09 Feb    60  learn from
VCT 2025: Americas Kickoff       17 Jan   09 Feb    56  learn from
VCT 2025: Pacific Kickoff        18 Jan   09 Feb    63  learn from
Valorant Masters Bangkok 2025    20 Feb   02 Mar    42  learn from
VCT 2025: China Stage 1          13 Mar   04 May   105  learn from
VCT 2025: Americas Stage 1       22 Mar   05 May   104  learn from
VCT 2025: Pacific Stage 1        22 Mar   11 May   107  learn from
VCT 2025: EMEA Stage 1           26 Mar   18 May   104  learn from
Valorant Masters Toronto 2025    07 Jun   22 Jun    59  learn from
VCT 2025: China Stage 2          03 Jul   24 Aug   110     test on
VCT 2025: Pacific Stage 2        15 Jul   31 Aug   103     test on
VCT 2025: EMEA Stage 2           16 Jul   31 Aug   110     tes

## 2. Where the validation set goes

The only clean gap in the learning half leaves about 59 maps, a margin of error of about
±13 points: too blunt to compare models. Instead, cut the matches into six chunks in date
order and fit five times, each time learning from everything before a chunk and checking on
that chunk:

| Fold | Learns from | Checked on |
|---|---|---|
| 1 | chunk 1 | chunk 2 |
| 2 | chunks 1–2 | chunk 3 |
| 3 | chunks 1–3 | chunk 4 |
| 4 | chunks 1–4 | chunk 5 |
| 5 | chunks 1–5 | chunk 6 |

That checks 629 maps. They come from only 246 matches, and in 53% of matches one team wins
every map, so they aren't independent: the honest margin of error is **±4 to 6 points**. That
can't separate models two points apart, but it can spot one that's worse on every fold.

Folds grow rather than slide, because with 756 maps, dropping January costs more than its
staleness. **Cuts fall between matches, never inside one**, so no match is half learned and
half checked.

In [3]:
# season_folds lives in src/dataset.py so notebook 05 uses exactly the
# same splitter rather than its own hand-copied version.
from dataset import N_FOLDS, season_folds

summary = []
for fold, (learn_rows, check_rows) in enumerate(season_folds(learn), start=1):
    a, b = learn.loc[learn_rows], learn.loc[check_rows]
    summary.append({
        "fold": fold,
        "learn maps": len(a),
        "learn matches": a["match_id"].nunique(),
        "check maps": len(b),
        "check matches": b["match_id"].nunique(),
        "checked on": f"{b['played_at'].min():%d %b} - {b['played_at'].max():%d %b}",
        "overlap": len(set(a["match_id"]) & set(b["match_id"])),
    })

print(pd.DataFrame(summary).to_string(index=False))
print(f"\ntotal maps used for checking: {sum(row['check maps'] for row in summary):,}")

 fold  learn maps  learn matches  check maps  check matches      checked on  overlap
    1         127             50         136             50 25 Jan - 27 Feb        0
    2         263            100         124             49 28 Feb - 31 Mar        0
    3         387            149         122             49 31 Mar - 14 Apr        0
    4         509            198         116             49 14 Apr - 03 May        0
    5         625            247         131             49 03 May - 22 Jun        0



total maps used for checking: 629


## 3. The first model

Logistic regression on default settings, to get one honest number on the board before any
tuning. (The default already includes a middling L2 penalty, `C=1.0`, which holds
coefficients back from chasing noise.)

Both baselines are recalculated on each fold's own maps, so every comparison is like for
like.

In [4]:
from sklearn.linear_model import LogisticRegression

X_learn, y_learn = build_features(learn)

scores = []
for fold, (learn_rows, check_rows) in enumerate(season_folds(learn), start=1):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_learn.loc[learn_rows], y_learn.loc[learn_rows])

    truth = y_learn.loc[check_rows]
    checked = learn.loc[check_rows]

    # Baselines measured on these same maps, not borrowed from the whole season.
    always_a = (truth == 1).mean()

    # The picker rule has to answer on every map the model answers on, so it
    # falls back to "team A" on decider maps, where nobody picked and the rule
    # has no opinion. Without the fallback it would be scored on fewer maps
    # than the model, which is not a fair comparison.
    picked_by_a = checked["map_picked_by"].map({"team_a": 1, "team_b": 0})
    picker_rule = (picked_by_a.fillna(1) == truth).mean()

    scores.append({
        "fold": fold,
        "maps": len(truth),
        "deciders": int(picked_by_a.isna().sum()),
        "always A": always_a,
        "map picker": picker_rule,
        "model": model.score(X_learn.loc[check_rows], truth),
    })

results = pd.DataFrame(scores).set_index("fold")

as_percent = results.copy()
for column in ("always A", "map picker", "model"):
    as_percent[column] = (as_percent[column] * 100).round(1)
print(as_percent.to_string())
print(f"\nthe model was fitted on {X_learn.shape[1]} columns")
print("\naverage across folds:")
for column in ("always A", "map picker", "model"):
    print(f"  {column:<12} {results[column].mean() * 100:5.1f}%   "
          f"(fold range {results[column].min() * 100:.1f} - {results[column].max() * 100:.1f})")

      maps  deciders  always A  map picker  model
fold                                             
1      136        23      52.9        58.8   53.7
2      124        22      54.8        58.9   52.4
3      122        24      50.0        50.8   50.0
4      116        18      45.7        50.0   53.4
5      131        16      48.9        58.8   47.3

the model was fitted on 38 columns

average across folds:
  always A      50.5%   (fold range 45.7 - 54.8)
  map picker    55.5%   (fold range 50.0 - 58.9)
  model         51.4%   (fold range 47.3 - 53.7)


## 4. Why does it lose to the baseline?

The model scores **51.4%** against the picker rule's **55.5%**. Of its 38 columns, 27 are
agents (which notebook 03 found carry no signal), 10 are maps, and 1 is who picked the map,
the column the rule is built from. (38 rather than 39: Corrode wasn't played until July.)

The suspicion is that the one useful column is drowned out. Fit each group of columns on its
own:

In [5]:
agent_columns = [c for c in X_learn.columns if c.startswith("agent_")]
map_columns = [c for c in X_learn.columns if c.startswith("map_") and c != "map_picked_by_a"]
picker_column = ["map_picked_by_a"]

# Column counts are worked out rather than written in, so the labels cannot
# drift away from what was actually fitted.
groups = {
    name: columns
    for name, columns in [
        (f"picker only ({len(picker_column)})", picker_column),
        (f"picker + map ({len(picker_column + map_columns)})", picker_column + map_columns),
        (f"agents only ({len(agent_columns)})", agent_columns),
        (f"agents + map ({len(agent_columns + map_columns)})", agent_columns + map_columns),
        (f"everything ({len(X_learn.columns)})", list(X_learn.columns)),
    ]
}


def score_across_folds(columns, C=1.0):
    """Average accuracy over the five folds, using only these columns."""
    per_fold = []
    for learn_rows, check_rows in season_folds(learn):
        model = LogisticRegression(C=C, max_iter=5000)
        model.fit(X_learn.loc[learn_rows, columns], y_learn.loc[learn_rows])
        per_fold.append(model.score(X_learn.loc[check_rows, columns], y_learn.loc[check_rows]))
    return np.array(per_fold)


comparison = []
for name, columns in groups.items():
    folds_scored = score_across_folds(columns)
    comparison.append({
        "features": name,
        "mean %": round(folds_scored.mean() * 100, 1),
        "worst fold": round(folds_scored.min() * 100, 1),
        "best fold": round(folds_scored.max() * 100, 1),
    })

print(pd.DataFrame(comparison).to_string(index=False))
print(f"\nmap-picker baseline on the same folds: {results['map picker'].mean() * 100:.1f}%")

         features  mean %  worst fold  best fold
  picker only (1)    54.4        50.0       58.9
picker + map (11)    53.5        49.1       60.3
 agents only (27)    48.0        42.7       50.7
agents + map (37)    49.6        45.8       53.4
  everything (38)    51.4        47.3       53.7

map-picker baseline on the same folds: 55.5%


**Who picked alone scores 54.4%**, within a point of the rule. Adding the other 37 columns
drops it to 51.4%, so the agents cost about three points. **Agents alone get 48.0% and never
beat 50.7% on any fold.**

That doesn't mean picking an agent makes a team lose. The model finds patterns in its
training maps that don't survive into the next part of the season: with 27 agent columns
and as few as 127 training maps, coincidences are easy to find, and applying them makes it
confidently wrong.

## 5. Does stronger regularisation rescue it?

Smaller `C` squeezes the coefficients harder. If the agents are noise, they should shrink
faster than the real signal.

In [6]:
sweep = []
for C in [0.001, 0.01, 0.03, 0.1, 0.3, 1.0, 10.0, 100.0]:
    folds_scored, agent_sizes, picker_sizes = [], [], []
    for learn_rows, check_rows in season_folds(learn):
        model = LogisticRegression(C=C, max_iter=5000)
        model.fit(X_learn.loc[learn_rows], y_learn.loc[learn_rows])
        folds_scored.append(model.score(X_learn.loc[check_rows], y_learn.loc[check_rows]))

        coefficients = pd.Series(model.coef_[0], index=X_learn.columns)
        agent_sizes.append(coefficients[agent_columns].abs().mean())
        picker_sizes.append(abs(coefficients["map_picked_by_a"]))

    sweep.append({
        "C": C,
        "mean %": round(np.mean(folds_scored) * 100, 1),
        "worst": round(min(folds_scored) * 100, 1),
        "best": round(max(folds_scored) * 100, 1),
        "avg |agent coef|": round(np.mean(agent_sizes), 3),
        "|picker coef|": round(np.mean(picker_sizes), 3),
    })

print(pd.DataFrame(sweep).to_string(index=False))
print(f"\npicker-only model: {score_across_folds(picker_column).mean() * 100:.1f}%"
      f"      map-picker baseline: {results['map picker'].mean() * 100:.1f}%")

      C  mean %  worst  best  avg |agent coef|  |picker coef|
  0.001    49.6   45.7  54.8             0.002          0.015
  0.010    52.6   49.1  55.7             0.019          0.086
  0.030    53.5   47.8  56.6             0.045          0.139
  0.100    51.4   48.1  55.6             0.099          0.183
  0.300    51.2   48.1  54.8             0.169          0.202
  1.000    51.4   47.3  53.7             0.249          0.209
 10.000    51.6   47.3  53.4             0.343          0.202
100.000    51.3   47.3  54.3             0.368          0.198

picker-only model: 54.4%      map-picker baseline: 55.5%


It helps a little (51.4% → 53.5% at `C=0.03`), but never reaches the 54.4% you get by dropping
the agents. At the default, the average agent coefficient (0.249) is bigger than the picker's
(0.209): 27 noise columns each weighted more than the one real one.

The 53.5% is the best of eight tries, inside the margin of error, so it doesn't make 0.03 the
"right" setting. What holds at every setting is the broader claim: **no `C` lets the agent
columns beat the model without them.**

## What this notebook found

1. **Agent picks don't predict the winner:** 51.4% against the picker rule's 55.5%.
2. **They actively cost accuracy:** without them, 54.4%; on their own, 48.0%.
3. **Tuning doesn't fix it.**
4. **This was the expected result.** This is the weaker of the plan's two models; the next
   notebooks add each player's form from *earlier* matches.

The map pool changed at the split (Corrode arrived; Fracture, Pearl and Split left), so the two
halves produced different columns. `build_features` now takes one fixed column list for the
whole season. The test half is still untouched.